<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/02_clasificacion_efficientnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 2 — Clasificación con EfficientNetB0

Clasificación binaria: `0 = sin paneles`, `1 = con paneles`. La etiqueta se deriva de la presencia de al menos una caja YOLO.

En Colab, el setup monta Google Drive. Cada variante se guarda con el formato común del equipo en `Mi unidad/PDI-TP/resultados/efficientnetb0/<modo>/`: pesos (`modelo.keras`), `metricas.json`, `predicciones_val.csv` (predicción contra el real) y `config.json`.

Se comparan tres configuraciones de la misma red:
1. Pesos ImageNet con extractor congelado.
2. Pesos ImageNet con fine-tuning del extractor.
3. Pesos aleatorios (`weights=None`).

La clase positiva se define por la presencia de anotaciones, no por el valor de la clase YOLO: en `solarpanels_hd` los IDs 0/1/2 son la confianza de la anotación (ver `docs/criterios_datos.md`).

**Fine-tuning** (esquema de la consigna): primero se entrena solo la cabeza con el extractor congelado (`Adam(1e-3)`) y después se descongela el extractor y se sigue con `Adam(1e-5)`. Todas las variantes usan **pesos de clase** calculados con train (desbalance 85/15).

In [ ]:
# Setup
import os, sys, json, random, subprocess
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

if 'google.colab' in sys.modules:
    repo = Path('/content/PDI-TP')
    if repo.is_dir():
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', 'https://github.com/candeperles/PDI-TP.git', str(repo)], check=True)
    # TensorFlow y versiones compatibles; no usar requirements.txt (sus versiones de NumPy rompen TF).
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-training.txt')],
                   check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pdi_tp' / 'entorno.py').is_file())
os.chdir(repo)
REPO = str(repo)
sys.path.insert(0, REPO)
from pdi_tp import entorno, datos, resultados
rutas = entorno.preparar()
print('Repositorio:', REPO)
print('Imágenes:', rutas.imagenes)


In [ ]:
# Dependencias
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Split y etiquetas binarias
def nombres_split(path):
    return [Path(x.strip()).name for x in Path(path).read_text().splitlines() if x.strip()]

def etiqueta(nombre):
    label = rutas.labels / f'{Path(nombre).stem}.txt'
    return int(label.read_text().strip() != '')

def preparar_lista(split_path):
    nombres = nombres_split(split_path)
    paths = [str(rutas.imagenes / nombre) for nombre in nombres]
    labels = [etiqueta(nombre) for nombre in nombres]
    return np.array(paths), np.array(labels, dtype=np.float32)

train_paths, y_train = preparar_lista(Path(REPO) / 'splits' / 'train.txt')
val_paths, y_val = preparar_lista(Path(REPO) / 'splits' / 'val.txt')
print('Train:', len(train_paths), 'positivos:', int(y_train.sum()))
print('Val:', len(val_paths), 'positivos:', int(y_val.sum()))
# Cada clase pesa en la loss en proporción inversa a su frecuencia (calculado solo con train)
_, cantidades = np.unique(y_train.astype(int), return_counts=True)
CLASS_WEIGHT = {c: len(y_train) / (2 * n) for c, n in enumerate(cantidades)}
print('Imágenes por clase (0 = sin, 1 = con):', dict(enumerate(cantidades)))
print('Pesos de clase:', CLASS_WEIGHT)

In [ ]:
# tf.data
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def cargar(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
], name='augmentation')

def dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = dataset(train_paths, y_train, training=True)
val_ds = dataset(val_paths, y_val)

In [ ]:
# Modelo EfficientNetB0
def construir_modelo(weights='imagenet', congelado=True):
    base = keras.applications.EfficientNetB0(
        include_top=False, weights=weights, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = not congelado
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = augment(inputs)
    # Con ImageNet se mantiene BatchNorm en inferencia; desde cero debe aprender sus estadisticas.
    # EfficientNet incluye su propio preprocesamiento.
    x = base(x, training=False) if weights == 'imagenet' else base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs), base

def entrenar(nombre, weights, fine_tune=False, epochs_head=10, epochs_ft=10):
    # Con ImageNet la etapa 1 entrena SOLO la cabeza (extractor congelado), también cuando después hay
    # fine-tuning; con pesos aleatorios se entrena toda la red desde el inicio.
    congelado = weights == 'imagenet'
    model, base = construir_modelo(weights=weights, congelado=congelado)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs_head, callbacks=callbacks,
                        class_weight=CLASS_WEIGHT)

    if fine_tune and weights == 'imagenet':
        # Segunda etapa: fine-tuning con learning rate bajo.
        base.trainable = True
        model.compile(optimizer=keras.optimizers.Adam(1e-5),
                      loss='binary_crossentropy', metrics=['accuracy'])
        history_ft = model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft, callbacks=callbacks,
                               class_weight=CLASS_WEIGHT)
        for key, values in history_ft.history.items():
            history.history.setdefault(key, []).extend(values)
    return model, history

## Entrenamiento

Ejecutar las variantes por separado si la memoria de la GPU es limitada. Los nombres quedan listos para comparar con MobileNet.

In [ ]:
import pandas as pd

MODELO = 'efficientnetb0'  # carpeta en Drive: resultados/efficientnetb0/<modo>/
REPETIR = False  # True vuelve a entrenar modos que ya tienen resultados completos (archiva los anteriores)

# modo -> (weights, fine_tune)
experimentos = {
    "imagenet_congelado": ('imagenet', False),
    "imagenet_finetuning": ('imagenet', True),
    "aleatorio": (None, False),
}

modelos = {}
historias = {}
carpetas = {}
for nombre, (weights, congelado) in experimentos.items():
    carpeta = resultados.ruta_corrida(rutas, MODELO, nombre)
    if resultados.corrida_completa(carpeta) and not REPETIR:
        print('Ya guardado, se saltea:', carpeta)
        continue
    carpetas[nombre] = resultados.carpeta_corrida(rutas, MODELO, nombre, reiniciar=REPETIR)
    print('\n===', nombre, '===')
    # Para el experimento de fine-tuning, entrenar() hace ambas etapas.
    modelo, historia = entrenar(nombre, weights, fine_tune=congelado, epochs_head=10, epochs_ft=10)
    modelos[nombre] = modelo
    historias[nombre] = historia.history
    modelo.save(carpetas[nombre] / 'modelo.keras')
    pd.DataFrame(historia.history).to_csv(carpetas[nombre] / 'historia.csv', index_label='epoca')

# Si se quiere ejecutar solo una variante, comentar el bucle y usar, por ejemplo:
# modelo, historia = entrenar('efficientnet_imagenet_congelado', 'imagenet', fine_tune=False)

In [ ]:
# Métricas sobre validación
def evaluar(nombre, model):
    prob = model.predict(val_ds, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    print(nombre)
    print(classification_report(y_val, pred, target_names=['sin_paneles', 'con_paneles'], zero_division=0))
    # Las métricas que se registran las calcula guardar_corrida, igual para todos los modelos.
    metricas = resultados.guardar_corrida(
        rutas, carpetas[nombre], archivos=val_paths, probabilidad=prob,
        params={'red': 'EfficientNetB0', 'weights': experimentos[nombre][0],
                'fine_tune': experimentos[nombre][1], 'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE,
                'augmentation': True, 'class_weight': CLASS_WEIGHT, 'epocas': [10, 10],
                'esquema_fine_tuning': 'cabeza con extractor congelado (1e-3) y luego extractor completo (1e-5)'},
        umbral=0.5,
    )
    print('Matriz de confusión (filas = real [sin, con]):\n', np.array(metricas['matriz_confusion']))
    return metricas

metricas_efficientnet = {nombre: evaluar(nombre, model) for nombre, model in modelos.items()}
metricas_efficientnet

In [ ]:
# Curvas de entrenamiento
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for nombre, h in historias.items():
    axes[0].plot(h['loss'], label=nombre)
    axes[1].plot(h['val_accuracy'], label=nombre)
axes[0].set_title('Loss'); axes[0].set_xlabel('Época'); axes[0].legend()
axes[1].set_title('Accuracy de validación'); axes[1].set_xlabel('Época'); axes[1].legend()
plt.show()